In [1]:
# DIAGNOSTIKA — Qashqadaryo HLS T41SPC may 2026
import ee
ee.Initialize(project='carbon-science-461016-q2')

col = (ee.ImageCollection('NASA/HLS/HLSL30/v002')
       .filter(ee.Filter.stringContains('system:index', 'T41SPC'))
       .filterDate('2026-05-01', '2026-06-01'))

n = col.size().getInfo()
print(f'Tasvirlar: {n} ta\n')

ids = col.aggregate_array('system:index').getInfo()
times = col.aggregate_array('system:time_start').getInfo()

import datetime
for idx, t in zip(ids, times):
    dt = datetime.datetime.utcfromtimestamp(t/1000)
    print(f'  {idx}  → UTC {dt}')

# Mosaic test (BUG REPRO)
distinct_dates = (col.aggregate_array('system:time_start')
                  .map(lambda t: ee.Date(t).format('YYYY-MM-dd'))
                  .distinct())

def best_per_date_BUGGY(date_str):
    date = ee.Date(date_str)
    daily = col.filterDate(date, date.advance(1, 'day'))
    return daily.mosaic().set('system:time_start', date.millis())   # BUG

def best_per_date_FIXED(date_str):
    date = ee.Date(date_str)
    daily = col.filterDate(date, date.advance(1, 'day'))
    actual_time = daily.first().get('system:time_start')
    return daily.mosaic().set('system:time_start', actual_time)

# Ikkalasini taqqoslash
mosaic_buggy = ee.ImageCollection(distinct_dates.map(best_per_date_BUGGY))
mosaic_fixed = ee.ImageCollection(distinct_dates.map(best_per_date_FIXED))

print('\nBUGGY versiya time_start:')
for t in mosaic_buggy.aggregate_array('system:time_start').getInfo():
    dt = datetime.datetime.utcfromtimestamp(t/1000)
    print(f'   {dt} UTC')   # ← 00:00:00 UTC chiqadi

print('\nFIXED versiya time_start:')
for t in mosaic_fixed.aggregate_array('system:time_start').getInfo():
    dt = datetime.datetime.utcfromtimestamp(t/1000)
    print(f'   {dt} UTC')   # ← 06:XX UTC chiqadi (real overpass)

Tasvirlar: 8 ta

  T41SPC_20260506T062339  → UTC 2026-05-06 06:23:39
  T41SPC_20260507T061710  → UTC 2026-05-07 06:17:10
  T41SPC_20260514T062310  → UTC 2026-05-14 06:23:10
  T41SPC_20260515T061718  → UTC 2026-05-15 06:17:18
  T41SPC_20260522T062322  → UTC 2026-05-22 06:23:22
  T41SPC_20260523T061657  → UTC 2026-05-23 06:16:57
  T41SPC_20260530T062307  → UTC 2026-05-30 06:23:07
  T41SPC_20260531T061708  → UTC 2026-05-31 06:17:08

BUGGY versiya time_start:
   2026-05-06 00:00:00 UTC
   2026-05-07 00:00:00 UTC
   2026-05-14 00:00:00 UTC
   2026-05-15 00:00:00 UTC
   2026-05-22 00:00:00 UTC
   2026-05-23 00:00:00 UTC
   2026-05-30 00:00:00 UTC
   2026-05-31 00:00:00 UTC

FIXED versiya time_start:
   2026-05-06 06:23:39 UTC
   2026-05-07 06:17:10 UTC
   2026-05-14 06:23:10 UTC
   2026-05-15 06:17:18 UTC
   2026-05-22 06:23:22 UTC
   2026-05-23 06:16:57 UTC
   2026-05-30 06:23:07 UTC
   2026-05-31 06:17:08 UTC


In [2]:
# Test — Landsat path ham ishlamoqdami?
from sebal_gee_v4.preprocessing import build_collection, _best_per_date_factory
import ee
ee.Initialize(project='carbon-science-461016-q2')

# Test 1: Factory ishlaydimi?
import datetime
col = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
       .filterDate('2026-05-01', '2026-06-01')
       .filter(ee.Filter.eq('WRS_PATH', 154))
       .filter(ee.Filter.eq('WRS_ROW', 32)))

best_per_date_fn = _best_per_date_factory(col)
dates = (col.aggregate_array('system:time_start')
         .map(lambda t: ee.Date(t).format('YYYY-MM-dd'))
         .distinct())

mosaic_col = ee.ImageCollection(dates.map(best_per_date_fn))
times = mosaic_col.aggregate_array('system:time_start').getInfo()

print('Landsat mosaic vaqtlari:')
for t in times:
    dt = datetime.datetime.utcfromtimestamp(t/1000)
    print(f'  {dt} UTC')
    
# Kutilgan: 05:30 UTC atrofida — quyosh chiqqan vaqt

Landsat mosaic vaqtlari:
  2026-05-01 06:04:47.732000 UTC
  2026-05-17 06:04:30.827000 UTC
